# Stimulus Spending Parser — ARRA & CARES Act

Reads directly from CBO PDFs — no hardcoded values. All figures are extracted from
the actual tables in the PDFs. Assertions verify extracted numbers against CBO's
own published headline totals before writing output.

**Setup — download these two PDFs into this directory:**
- `arra_cost_estimate.pdf` → https://www.cbo.gov/publication/41762
- `hr748.pdf` → https://www.cbo.gov/system/files/2020-04/hr748.pdf

**Output schema:** `law, fiscal_year, category, amount_billions, pct_gdp`

## Cell 1 — Imports and helpers

In [63]:
import pdfplumber
import pandas as pd
import openpyxl
import re

ARRA_PDF   = 'arra_cost_estimate.pdf'
CARES_PDF  = 'hr748.pdf'
OMB_FILE   = 'BUDGET-2026-HIST.xlsx'
OUTPUT_CSV = '../public/stimulus_spending.csv'

def to_num(s):
    """CBO cell -> float. '*' = 0, parentheses = negative, None = unparseable."""
    s = str(s or '').strip().replace(',', '')
    if s in ('', '*', '**', 'n.a.', '-'): return 0.0
    if s.startswith('(') and s.endswith(')'): return -float(s[1:-1])
    try: return float(s)
    except: return None

def find_year_cols(table, year_range):
    """Scan table rows for year integers; return {col_index: year} from first matching row."""
    for row in table:
        col_map = {}
        for j, cell in enumerate(row):
            try:
                yr = int(str(cell or '').strip())
                if year_range[0] <= yr <= year_range[1]:
                    col_map[j] = yr
            except:
                pass
        if col_map:
            return col_map
    return {}

def row_vals(row, col_map, years):
    """Extract {year: float} from a table row using col_map, filtering to requested years."""
    return {yr: to_num(row[j])
            for j, yr in col_map.items()
            if yr in years and to_num(row[j]) is not None}

print('Ready.')

Ready.


## Cell 2 — Inspect ARRA PDF

Run this to see which pages hold the summary cost table and what the row labels look like.

In [64]:
with pdfplumber.open(ARRA_PDF) as pdf:
    print(f'ARRA: {len(pdf.pages)} pages')
    for i, page in enumerate(pdf.pages[:8]):
        text = page.extract_text() or ''
        if any(k in text for k in ['Division A', 'Division B', 'Total Changes', 'Revenues']):
            print(f'\n--- Page {i+1} ---')
            print(text[:600])
            for t in page.extract_tables():
                col_map = find_year_cols(t, (2009, 2019))
                if not col_map:
                    continue
                print(f'  Table: {len(t)} rows, year cols: {col_map}')
                for row in t:
                    label = str(row[0] or '').strip()
                    if label:
                        print(f'    [{label[:60]}]', [row[j] for j in col_map])

ARRA: 8 pages

--- Page 3 ---
TABLE 1. SUMMARY OF ESTIMATED COST OF THE CONFERENCE AGREEMENT FOR H.R. 1, THE
AMERICAN RECOVERY AND REINVESTMENT ACT OF 2009, AS POSTED ON THE WEB SITE OF
THE HOUSE COMMITTEE ON RULES
By Fiscal Year, in Billions of Dollars
2009-
2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2019
DIVISION A—APPROPRIATIONS a
Estimated Budget Authority 288.7 7.1 4.6 3.6 2.5 1.1 1.1 1.1 1.1 0.5 0 311.2
Estimated Outlays 34.8 110.7 76.3 38.1 22.9 12.8 7.0 3.1 1.6 0.8 0.1 308.3
DIVISION A—REVENUES
Estimated Revenues * * * * * * * * * * * -0.1
DIVISION B—DIRECT SPENDING
Estimated Budget Authority 90.3 107.6

--- Page 4 ---
Table 2. ESTIMATED COST OF THE CONFERENCE AGREEMENT FOR H.R. 1, THE AMERICAN RECOVERY AND REINVESTMENT ACT OF 2009,
AS POSTED ON THE WEBSITE OF THE HOUSE COMMITTEE ON RULES
By Fiscal Year, Millions of Dollars
Total
2009 -
2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2019
Discretionary Spending 1/
Division A
Title I - Agriculture, Rural
Devel

## Cell 3 — Parse ARRA

Source: CBO pub. 41762 — H.R.1 conference agreement, Feb 13 2009  
https://www.cbo.gov/publication/41762

Looks for:
- **Division A outlays** — discretionary appropriations (infrastructure, grants, agencies)
- **Division B outlays** — mandatory/direct spending (UI, Medicaid, nutrition, refundable credits)
- **Total revenue reductions** — tax cuts (stored as positive)

Validation: 2009+2010 combined total should be near CBO headline of $185B + $399B = $584B.

In [65]:
ARRA_YEARS = [2009, 2010, 2011, 2012, 2013]

def parse_num_line(line, n_years):
    tokens = re.findall(r'-?\d+\.\d+|-?\d+', line)
    nums = []
    for t in tokens:
        try: nums.append(float(t))
        except: pass
    return nums if len(nums) >= n_years else []

discr_by_yr = {}
mand_by_yr  = {}
rev_by_yr   = {}

with pdfplumber.open(ARRA_PDF) as pdf:
    text = pdf.pages[2].extract_text() or ''
    print('Page 3 raw text:')
    print(text)

    lines = text.split('\n')
    current_section = None

    for line in lines:
        ll = line.strip().lower()

        if 'division a' in ll and 'appropriation' in ll:
            current_section = 'div_a'
            print('-> Section: Division A Appropriations')
        elif 'division a' in ll and 'revenue' in ll:
            current_section = 'div_a_rev'
        elif 'division b' in ll and 'direct spending' in ll:
            current_section = 'div_b_spending'
            print('-> Section: Division B Direct Spending')
        elif 'division b' in ll and 'revenue' in ll:
            current_section = 'div_b_revenue'
            print('-> Section: Division B Revenues')

        if 'estimated outlays' in ll and current_section == 'div_a' and not discr_by_yr:
            nums = parse_num_line(line, len(ARRA_YEARS))
            if nums:
                discr_by_yr = dict(zip(ARRA_YEARS, nums[:len(ARRA_YEARS)]))
                print(f'  Division A outlays: {discr_by_yr}')

        if 'estimated outlays' in ll and current_section == 'div_b_spending' and not mand_by_yr:
            nums = parse_num_line(line, len(ARRA_YEARS))
            if nums:
                mand_by_yr = dict(zip(ARRA_YEARS, nums[:len(ARRA_YEARS)]))
                print(f'  Division B outlays: {mand_by_yr}')

        if 'estimated revenues' in ll and current_section == 'div_b_revenue' and not rev_by_yr:
            nums = parse_num_line(line, len(ARRA_YEARS))
            if nums:
                rev_by_yr = {yr: abs(v) for yr, v in zip(ARRA_YEARS, nums[:len(ARRA_YEARS)])}
                print(f'  Revenue reductions: {rev_by_yr}')

assert discr_by_yr, 'FAIL: Division A outlays not found — check printed text above'
assert mand_by_yr,  'FAIL: Division B outlays not found — check printed text above'
assert rev_by_yr,   'FAIL: Revenue reductions not found — check printed text above'

combined_09_10 = sum(
    discr_by_yr.get(y, 0) + mand_by_yr.get(y, 0) + rev_by_yr.get(y, 0)
    for y in [2009, 2010]
)
print(f'\n2009+2010 combined: ${combined_09_10:.1f}B  (CBO headline: ~$584B)')
assert 450 < combined_09_10 < 750, f'FAIL: ${combined_09_10:.1f}B outside expected $450-$750B'
print('All assertions passed.')

arra_rows = []
for yr in ARRA_YEARS:
    arra_rows.append({'law': 'ARRA', 'fiscal_year': yr, 'category': 'Discretionary Outlays', 'amount_billions': discr_by_yr.get(yr, 0.0)})
    arra_rows.append({'law': 'ARRA', 'fiscal_year': yr, 'category': 'Mandatory Outlays',     'amount_billions': mand_by_yr.get(yr, 0.0)})
    arra_rows.append({'law': 'ARRA', 'fiscal_year': yr, 'category': 'Revenue Reductions',    'amount_billions': rev_by_yr.get(yr, 0.0)})

df_arra = pd.DataFrame(arra_rows)
totals  = df_arra.groupby(['law','fiscal_year'])['amount_billions'].sum().reset_index()
totals['category'] = 'Total Direct Cost'
df_arra = pd.concat([df_arra, totals], ignore_index=True)
print('\nARRA totals by year:')
print(df_arra[df_arra.category == 'Total Direct Cost'].to_string(index=False))

Page 3 raw text:
TABLE 1. SUMMARY OF ESTIMATED COST OF THE CONFERENCE AGREEMENT FOR H.R. 1, THE
AMERICAN RECOVERY AND REINVESTMENT ACT OF 2009, AS POSTED ON THE WEB SITE OF
THE HOUSE COMMITTEE ON RULES
By Fiscal Year, in Billions of Dollars
2009-
2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2019
DIVISION A—APPROPRIATIONS a
Estimated Budget Authority 288.7 7.1 4.6 3.6 2.5 1.1 1.1 1.1 1.1 0.5 0 311.2
Estimated Outlays 34.8 110.7 76.3 38.1 22.9 12.8 7.0 3.1 1.6 0.8 0.1 308.3
DIVISION A—REVENUES
Estimated Revenues * * * * * * * * * * * -0.1
DIVISION B—DIRECT SPENDING
Estimated Budget Authority 90.3 107.6 49.0 7.6 7.3 15.1 4.7 -4.7 -4.1 -1.9 -1.4 269.5
Estimated Outlays 85.3 108.6 49.9 8.1 7.4 15.1 4.7 -4.7 -4.1 -1.9 -1.4 267.0
DIVISION B—REVENUES
Estimated Revenues -64.8 -180.1 -8.2 10.0 2.7 5.5 7.1 5.8 5.1 5.0 0.1 -211.8
NET IMPACT ON THE DEFICIT
Net Increase or Decrease (-)
in the Deficit 184.9 399.4 134.4 36.1 27.6 22.4 4.7 -7.3 -7.5 -6.1 -1.4 787.2
a. Most of the spending for

## Cell 4 — Inspect CARES Act PDF

Run this to locate Table 2 (direct spending), Table 3 (revenues), Table 4 (discretionary).  
These are on the last few pages of hr748.pdf.

In [66]:
with pdfplumber.open(CARES_PDF) as pdf:
    print(f'CARES: {len(pdf.pages)} pages')
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ''
        if any(k in text for k in ['Table 2', 'Table 3', 'Table 4',
                                    'Total Changes in Direct Spending',
                                    'Total Changes in Revenues']):
            print(f'\n--- Page {i+1} ---')
            print(text[:400])
            for t in page.extract_tables():
                col_map = find_year_cols(t, (2020, 2030))
                if not col_map:
                    continue
                print(f'  Table: {len(t)} rows, year cols: {col_map}')
                for row in t:
                    label = str(row[0] or '').strip()
                    if label:
                        print(f'    [{label[:70]}]', [row[j] for j in col_map] if col_map else row)

CARES: 35 pages



--- Page 6 ---
Honorable Mike Enzi
Page 6
approach led to more informative estimates of the effects of those
provisions.
Estimated Federal Costs
In total, CBO and JCT estimate, the act will increase deficits by
$1.7 trillion over the 2020-2030 period; most of the budgetary effects occur
in 2020 and 2021. The provisions with the largest deficit effects are the
PPP, the recovery rebates, and the expansion of unemp

--- Page 24 ---
Honorable Mike Enzi
Page 24
Title VI, Miscellaneous Provisions
Section 6001 provides $10 billion in additional borrowing authority for the
U.S. Postal Service to help cover operating expenses. Such borrowing
authority can be used by the Postal Service to increase its spending without
a corresponding increase in offsetting receipts. Using information from the
Postal Service, CBO estimates that the 

--- Page 31 ---
Table 2. Changes in Direct Spending Under Division A of H.R. 748, the Coronavirus Aid, Relief, and Economic Security (CARES) Act, Public Law 116-136

## Cell 5 — Parse CARES Act

Source: CBO pub. 56334 — revised April 27 2020  
https://www.cbo.gov/publication/56334

Reads three tables from the PDF:
- **Table 2** total `Estimated Outlays` row → mandatory/direct spending by year
- **Table 3** `Total Changes in Revenues` row → revenue reductions by year (stored as positive)
- **Table 4** total outlays row → discretionary (Division B) by year

Validation checks against CBO p.2 headlines:
- Mandatory (Table 2): ~$988B over 2020-2030
- Revenue (Table 3): ~$408B over 2020-2030
- Discretionary (Table 4): ~$326B total

In [67]:
CARES_YEARS = [2020, 2021, 2022]

def parse_cares_line(line, years):
    tokens = re.findall(r'-?\d{1,4}(?:,\d{3})*(?:\.\d+)?', line)
    nums = []
    for t in tokens:
        try: nums.append(float(t.replace(',', '')))
        except: pass
    if len(nums) >= len(years):
        return dict(zip(years, nums[:len(years)]))
    return {}

outlays_by_yr = {}
revenue_by_yr = {}
discr_by_yr   = {}
cares_discr_by_yr = {}  # <-- ADD THIS

with pdfplumber.open(CARES_PDF) as pdf:

    page33_text = pdf.pages[32].extract_text() or ''
    lines = page33_text.split('\n')
    found_total_header = False
    for line in lines:
        if 'Total Changes in Direct Spending' in line:
            found_total_header = True
            continue
        if found_total_header and 'Estimated Outlays' in line and 'On-Budget' not in line and 'Off-Budget' not in line:
            vals = parse_cares_line(line, CARES_YEARS)
            if vals and max(vals.values(), default=0) > 100:
                outlays_by_yr = vals
                print(f'Table 2 grand total outlays: {outlays_by_yr}')
                break

    page34_text = pdf.pages[33].extract_text() or ''
    for line in page34_text.split('\n'):
        if line.strip().startswith('Total Changes in Revenues'):
            vals = parse_cares_line(line, CARES_YEARS)
            if vals:
                revenue_by_yr = {yr: abs(v) for yr, v in vals.items()}
                print(f'Table 3 revenues: {revenue_by_yr}')
                break

    page35_text = pdf.pages[34].extract_text() or ''
    found_total_header = False
    for line in page35_text.split('\n'):
        if 'Total Changes in Discretionary Spending' in line:
            found_total_header = True
            continue
        if found_total_header and 'Estimated Outlays' in line \
                and 'On-Budget' not in line and 'Off-Budget' not in line:
            vals = parse_cares_line(line, CARES_YEARS)
            if vals and max(vals.values(), default=0) > 50:
                discr_by_yr = vals
                cares_discr_by_yr = vals.copy()  # <-- ADD THIS
                print(f'Table 4 discretionary outlays: {discr_by_yr}')
                break

assert outlays_by_yr,     'FAIL: Table 2 grand total outlays not found'
assert revenue_by_yr,     'FAIL: Table 3 revenues not found'
assert cares_discr_by_yr, 'FAIL: Table 4 discretionary not found'

disc_total = sum(cares_discr_by_yr.values())
print(f'\nValidation (CBO pub. 56334 p.2 headlines):')
print(f'  Mandatory  2020=${outlays_by_yr.get(2020,0):.0f}B  2021=${outlays_by_yr.get(2021,0):.0f}B  (10yr ~$988B)')
print(f'  Revenue    2020=${revenue_by_yr.get(2020,0):.0f}B  2021=${revenue_by_yr.get(2021,0):.0f}B  (10yr ~$408B)')
print(f'  Discr      2020=${cares_discr_by_yr.get(2020,0):.0f}B  2021=${cares_discr_by_yr.get(2021,0):.0f}B  (total ~$326B)')

assert 800 < outlays_by_yr.get(2020, 0) < 1100
assert 300 < revenue_by_yr.get(2020, 0) < 700
assert 150 < disc_total < 450
print('All assertions passed.')

df_cares = pd.DataFrame([
    {'law': 'CARES Act', 'fiscal_year': yr, 'category': 'Mandatory Outlays',     'amount_billions': outlays_by_yr.get(yr, 0.0)}
    for yr in CARES_YEARS
] + [
    {'law': 'CARES Act', 'fiscal_year': yr, 'category': 'Discretionary Outlays', 'amount_billions': cares_discr_by_yr.get(yr, 0.0)}
    for yr in CARES_YEARS
] + [
    {'law': 'CARES Act', 'fiscal_year': yr, 'category': 'Revenue Reductions',    'amount_billions': revenue_by_yr.get(yr, 0.0)}
    for yr in CARES_YEARS
])
totals = df_cares.groupby(['law','fiscal_year'])['amount_billions'].sum().reset_index()
totals['category'] = 'Total Direct Cost'
df_cares = pd.concat([df_cares, totals], ignore_index=True)
print('\nCARES totals by year:')
print(df_cares[df_cares.category == 'Total Direct Cost'].to_string(index=False))

Table 2 grand total outlays: {2020: 938.0, 2021: 73.0, 2022: 3.0}
Table 3 revenues: {2020: 568.0, 2021: 240.0, 2022: 177.0}
Table 4 discretionary outlays: {2020: 99.0, 2021: 134.0, 2022: 58.0}

Validation (CBO pub. 56334 p.2 headlines):
  Mandatory  2020=$938B  2021=$73B  (10yr ~$988B)
  Revenue    2020=$568B  2021=$240B  (10yr ~$408B)
  Discr      2020=$99B  2021=$134B  (total ~$326B)
All assertions passed.

CARES totals by year:
      law  fiscal_year          category  amount_billions
CARES Act         2020 Total Direct Cost           1605.0
CARES Act         2021 Total Direct Cost            447.0
CARES Act         2022 Total Direct Cost            238.0


## Cell 6 — Load GDP from OMB Historical Tables (hist10z1)

Parses GDP (in millions) from BUDGET-2026-HIST.xlsx. Converted to billions for pct_gdp.

In [68]:
import ssl
import urllib.request
import io

# Bypass SSL verification (safe for public FRED data)
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

FRED_GDP_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=GDPA'

with urllib.request.urlopen(FRED_GDP_URL, context=ctx) as r:
    gdp_df = pd.read_csv(io.BytesIO(r.read()), parse_dates=['observation_date'])

gdp_df['year'] = gdp_df['observation_date'].dt.year
gdp_lookup = dict(zip(gdp_df['year'], gdp_df['GDPA']))

for y in [2008, 2009, 2010, 2011, 2012, 2013, 2014,2020, 2021, 2022]:
    g = gdp_lookup.get(y)
    print(f'  GDP {y}: ${g/1000:.2f}T' if g else f'  WARN: GDP {y} not found')

  GDP 2008: $14.77T
  GDP 2009: $14.48T
  GDP 2010: $15.05T
  GDP 2011: $15.60T
  GDP 2012: $16.25T
  GDP 2013: $16.88T
  GDP 2014: $17.61T
  GDP 2020: $21.38T
  GDP 2021: $23.73T
  GDP 2022: $26.05T


## Cell 7 — Combine, compute % of GDP, final checks, write output

In [69]:
df = pd.concat([df_arra, df_cares], ignore_index=True)

df['gdp_b']   = df['fiscal_year'].map(lambda y: gdp_lookup.get(y))
df['pct_gdp'] = (df['amount_billions'] / df['gdp_b'] * 100).round(2)
df = df.drop(columns=['gdp_b'])
df = df.sort_values(['law', 'fiscal_year', 'category']).reset_index(drop=True)

print('Final totals:')
print(df[df.category=='Total Direct Cost'][['law','fiscal_year','amount_billions','pct_gdp']].to_string(index=False))

def get_total(law, yr):
    rows = df[(df.law==law) & (df.fiscal_year==yr) & (df.category=='Total Direct Cost')]
    return rows['amount_billions'].values[0] if len(rows) else None

assert 100 < get_total('ARRA', 2009) < 300,        f'FAIL: ARRA 2009 = ${get_total("ARRA",2009):.0f}B'
assert 300 < get_total('ARRA', 2010) < 500,        f'FAIL: ARRA 2010 = ${get_total("ARRA",2010):.0f}B'
assert 1000 < get_total('CARES Act', 2020) < 2500, f'FAIL: CARES 2020 = ${get_total("CARES Act",2020):.0f}B'
print('All assertions passed.')

Final totals:
      law  fiscal_year  amount_billions  pct_gdp
     ARRA         2009            184.9     1.28
     ARRA         2010            399.4     2.65
     ARRA         2011            134.4     0.86
     ARRA         2012             56.2     0.35
     ARRA         2013             33.0     0.20
CARES Act         2020           1605.0     7.51
CARES Act         2021            447.0     1.88
CARES Act         2022            238.0     0.91
All assertions passed.


In [70]:
OUTPUT_CSV = '../public/stimulus_spending.csv'

df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {OUTPUT_CSV}  ({len(df)} rows)')
print(df.to_string(index=False))

Saved ../public/stimulus_spending.csv  (32 rows)
      law  fiscal_year              category  amount_billions  pct_gdp
     ARRA         2009 Discretionary Outlays             34.8     0.24
     ARRA         2009     Mandatory Outlays             85.3     0.59
     ARRA         2009    Revenue Reductions             64.8     0.45
     ARRA         2009     Total Direct Cost            184.9     1.28
     ARRA         2010 Discretionary Outlays            110.7     0.74
     ARRA         2010     Mandatory Outlays            108.6     0.72
     ARRA         2010    Revenue Reductions            180.1     1.20
     ARRA         2010     Total Direct Cost            399.4     2.65
     ARRA         2011 Discretionary Outlays             76.3     0.49
     ARRA         2011     Mandatory Outlays             49.9     0.32
     ARRA         2011    Revenue Reductions              8.2     0.05
     ARRA         2011     Total Direct Cost            134.4     0.86
     ARRA         2012 Discr